In [ ]:
pip install mne torch-geometric

In [ ]:


import os
from os.path import dirname, join as pjoin
import scipy as sp
import scipy.io as sio
from scipy import signal
import numpy as np
from matplotlib.pyplot import specgram
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal as sig
import networkx as nx
import torch as torch
from scipy.signal import welch
from scipy.stats import entropy
from sklearn.feature_selection import mutual_info_classif
from scipy.integrate import simpson
from sklearn.model_selection import KFold
from torch_geometric.loader import DataLoader
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, GATv2Conv, GAT, GraphNorm
from torch_geometric.nn import global_mean_pool
from torch import nn
from tqdm import tqdm
from torch_geometric.data import Data



In [ ]:
# --- Core Libraries ---
import numpy as np
import scipy.io as sio
from scipy import signal as sig
from scipy.integrate import simpson
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import networkx as nx
import os
from os.path import join as pjoin

# --- PyTorch & PyG Libraries ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, global_mean_pool, GraphNorm
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

# --- Scikit-learn for Evaluation ---
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, classification_report

# --- MNE for Topographical Plots (Crucial for Visualization) ---
import mne

# --- GPU Setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Matplotlib plotting style ---
plt.style.use('seaborn-v0_8-whitegrid')
print("All libraries imported and device is set.")

In [ ]:
def load_bci_competition_data(data_dir, subject_number):
    """
    Load BCI Competition IV Dataset 2a and convert to the expected format.
    
    Args:
        data_dir: Directory containing the .mat files
        subject_number: Subject number (1-9)
    
    Returns:
        subject_data: Dictionary with 'L' and 'R' keys containing trial data
    """
    mat_fname = pjoin(data_dir, f'A0{subject_number}T.mat')
    mat_contents = sio.loadmat(mat_fname, squeeze_me=True, struct_as_record=False)
    data = mat_contents['data']
    
    left_trials = []
    right_trials = []
    
    for session in data:
        # We are only interested in the training sessions which have labels
        if 'X' in session._fieldnames:
             eeg_data = session.X
             trial_positions = session.trial
             trial_labels = session.y
             
             # Use only the 22 EEG channels
             eeg_data = eeg_data[:, :22]
             
             # Extract trials (4 seconds * 250 Hz = 1000 samples)
             trial_length = 1000
             
             for start_pos, label in zip(trial_positions, trial_labels):
                 start_sample = int(start_pos)
                 end_sample = start_sample + trial_length
                 
                 if end_sample <= eeg_data.shape[0]:
                     trial_data = eeg_data[start_sample:end_sample, :]
                     
                     if label == 1: # Left hand
                         left_trials.append(trial_data)
                     elif label == 2: # Right hand
                         right_trials.append(trial_data)
    
    # Convert to the expected format (list of numpy arrays)
    subject_data = {
        'L': left_trials,
        'R': right_trials
    }
    
    return subject_data

# --- Example Usage ---
# Note: Adjust the path to where your data is stored on Kaggle.
data_dir = '/kaggle/input/bci-competition-iv-data-sets-2a'
subject_1_data_raw = load_bci_competition_data(data_dir, 1)

print(f"Loaded data for Subject 1.")
print(f"Number of Left hand MI trials: {len(subject_1_data_raw['L'])}")
print(f"Number of Right hand MI trials: {len(subject_1_data_raw['R'])}")
print(f"Shape of a single trial: {subject_1_data_raw['L'][0].shape}")
print(f"(Time Samples, EEG Channels)")

In [ ]:
def bandpass(data: np.ndarray, edges: list[float], sample_rate: float, poles: int = 5):
    """Apply a bandpass filter to the data."""
    sos = sig.butter(poles, edges, 'bandpass', fs=sample_rate, output='sos')
    filtered_data = sig.sosfiltfilt(sos, data, axis=0)
    return filtered_data

# --- Create a copy for processing ---
subject_1_data_processed = {
    'L': [trial.copy() for trial in subject_1_data_raw['L']],
    'R': [trial.copy() for trial in subject_1_data_raw['R']]
}

# --- Apply Bandpass Filter ---
fs = 250  # Sampling frequency
freq_band = [8, 30] # Mu and Beta rhythm band
for class_key in subject_1_data_processed:
    for i in range(len(subject_1_data_processed[class_key])):
        subject_1_data_processed[class_key][i] = bandpass(
            subject_1_data_processed[class_key][i], freq_band, fs
        )

print("Applied bandpass filter (8-30 Hz) to all trials.")

# --- Visualization of Filtering ---
# Select one trial and one channel to visualize
trial_raw = subject_1_data_raw['L'][0]
trial_filtered = subject_1_data_processed['L'][0]
channel_idx = 8 # C3 electrode

# 1. Time Domain Plot
time = np.arange(trial_raw.shape[0]) / fs
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(time, trial_raw[:, channel_idx], label='Raw EEG', color='red', alpha=0.8)
ax1.set_title(f'Time Domain Signal (Channel C3)')
ax1.set_ylabel('Amplitude (µV)')
ax1.legend()
ax2.plot(time, trial_filtered[:, channel_idx], label='Filtered (8-30Hz) EEG', color='blue')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Amplitude (µV)')
ax2.legend()
plt.suptitle('Effect of Bandpass Filtering', fontsize=16)
plt.show()


# 2. Frequency Domain Plot (Power Spectral Density)
freqs_raw, psd_raw = sig.welch(trial_raw[:, channel_idx], fs)
freqs_filt, psd_filt = sig.welch(trial_filtered[:, channel_idx], fs)
fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(freqs_raw, psd_raw, label='Raw EEG Spectrum', color='red', alpha=0.7)
ax.semilogy(freqs_filt, psd_filt, label='Filtered EEG Spectrum', color='blue', linewidth=2)
ax.axvspan(freq_band[0], freq_band[1], color='green', alpha=0.2, label='Band of Interest')
ax.set_title('Power Spectral Density (PSD)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power/Frequency (dB/Hz)')
ax.set_xlim(0, 50)
ax.legend()
plt.show()

In [ ]:
def plvfcn(eeg_data):
    """
    Calculates the Phase Locking Value (PLV) matrix for a single trial.
    """
    num_channels = eeg_data.shape[1]
    num_samples = eeg_data.shape[0]
    analytic_signal = sig.hilbert(eeg_data, axis=0)
    instantaneous_phase = np.angle(analytic_signal)
    
    plv_matrix = np.zeros((num_channels, num_channels))
    for i in range(num_channels):
        for j in range(i + 1, num_channels):
            phase_diff = instantaneous_phase[:, i] - instantaneous_phase[:, j]
            plv = np.abs(np.sum(np.exp(1j * phase_diff)) / num_samples)
            plv_matrix[i, j] = plv
            plv_matrix[j, i] = plv
            
    return plv_matrix

def compute_all_plv_matrices(filtered_data):
    """
    Computes PLV matrices for all trials in the dataset.
    """
    all_plv = {'L': [], 'R': []}
    for class_key in filtered_data:
        for trial_data in tqdm(filtered_data[class_key], desc=f'Computing PLV for {class_key} trials'):
            plv_matrix = plvfcn(trial_data)
            all_plv[class_key].append(plv_matrix)
    return all_plv

# --- Compute PLV for our processed data ---
subject_1_plv = compute_all_plv_matrices(subject_1_data_processed)

# --- Average PLV matrices for visualization ---
avg_plv_L = np.mean(subject_1_plv['L'], axis=0)
avg_plv_R = np.mean(subject_1_plv['R'], axis=0)

print(f"\nComputed PLV matrices. Average Left PLV matrix shape: {avg_plv_L.shape}")

# --- VISUALIZATION 1: PLV Matrix Heatmap ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
# Plotting code for heatmaps remains the same...
sns.heatmap(avg_plv_L, ax=ax1, cmap='viridis', vmin=0, vmax=np.max([avg_plv_L, avg_plv_R]))
ax1.set_title('Average PLV Matrix (Left Hand MI)')
ax1.set_xlabel('Channel Index'); ax1.set_ylabel('Channel Index')
sns.heatmap(avg_plv_R, ax=ax2, cmap='viridis', vmin=0, vmax=np.max([avg_plv_L, avg_plv_R]))
ax2.set_title('Average PLV Matrix (Right Hand MI)')
ax2.set_xlabel('Channel Index');
plt.suptitle('All-to-All Functional Connectivity', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()


# --- VISUALIZATION 2: Topographical Connectivity Plot (Final Corrected Version) ---
ch_names = ['FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6', 'C5', 'C3', 'C1', 
            'Cz', 'C2', 'C4', 'C6', 'CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6', 'Fz']
info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types='eeg')
montage = mne.channels.make_standard_montage('standard_1005')
info.set_montage(montage)

# This is our new, robust plotting function
def plot_brain_connectivity(plv_matrix, ax, title, threshold=0.8):
    """
    Plots brain connectivity on a 2D sensor layout.
    """
    # Use mne.viz.plot_sensors to draw the channel locations on the given axes
    # This is the corrected line that fixes the TypeError
    mne.viz.plot_sensors(info, kind='topomap', ch_type='eeg', show_names=True, show=False)
    
    # Get the 2D sensor positions from the info object's montage
    ch_coords = info.get_montage().get_positions()['ch_pos']
    pos = np.array([ch_coords[ch] for ch in info['ch_names']])

    # Find connections above the threshold
    source, target = np.where(plv_matrix > threshold)
    
    # Plot the connections
    for i in range(len(source)):
        # We only plot each connection once
        if source[i] > target[i]:
            x1, y1 = pos[source[i], :2]
            x2, y2 = pos[target[i], :2]
            
            weight = plv_matrix[source[i], target[i]]
            
            ax.plot([x1, x2], [y1, y2], 
                    color='red', 
                    linewidth=0.5 + 2 * weight,
                    alpha=0.5 + 0.5 * weight)

    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)


# Plotting the brain networks
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), subplot_kw={'aspect': 'equal'})
plot_brain_connectivity(avg_plv_L, ax1, 'Connectivity during Left Hand MI')
plot_brain_connectivity(avg_plv_R, ax2, 'Connectivity during Right Hand MI')
plt.suptitle('Brain Network Visualization (PLV > 0.35)', fontsize=16)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()


print("MONTAGE VISUALIZATION CODE DIDNT WORK HENCE THE GRAPH AND THE ELECTRODE PLACEMENT IS NOT SHOW TOGETHER")

In [ ]:
from scipy.signal import welch
from tqdm import tqdm
import numpy as np

def extract_mff_features(filtered_data, fs=250):
    """
    Extracts MFF (Mean Frequency Feature) for each EEG channel in each trial.
    
    Args:
        filtered_data (dict): {'L': [array(trials)], 'R': [array(trials)]}
                              each array is (time_points, channels)
        fs (int): Sampling frequency
        
    Returns:
        features (dict): {'L': [ (channels, 1) arrays ], 'R': [ (channels, 1) arrays ]}
    """
    features = {'L': [], 'R': []}
    
    for key in filtered_data.keys():
        for trial_data in tqdm(filtered_data[key], desc=f'Computing MFF for {key}'):
            num_channels = trial_data.shape[1]
            trial_features = np.zeros((num_channels, 1))  # MFF per channel
            
            for ch_idx in range(num_channels):
                signal = trial_data[:, ch_idx]
                
                # Compute Power Spectral Density
                freqs, psd = welch(signal, fs=fs, nperseg=min(256, len(signal)))
                
                # Mean Frequency: sum(f * P(f)) / sum(P(f))
                mff = np.sum(freqs * psd) / np.sum(psd)
                trial_features[ch_idx, 0] = mff
            
            features[key].append(trial_features)
    
    return features
subject_1_features = extract_mff_features(subject_1_data_processed)
print(f"Example MFF shape for one trial (channels, features): {subject_1_features['L'][0].shape}")


In [ ]:
from torch_geometric.utils import dense_to_sparse

# --- 1. Assemble Data into PyTorch Geometric's format ---

# We will create a fully connected graph for each trial initially.
# The edge_weights will be the PLV values. The GAT's attention
# mechanism will then learn which of these connections are important.

data_list = []
for i in range(len(subject_1_plv['L'])):
    x = torch.tensor(subject_1_features['L'][i], dtype=torch.float)  # (22, 1)
    adj = torch.tensor(subject_1_plv['L'][i], dtype=torch.float)
    edge_index, edge_attr = dense_to_sparse(adj)
    y = torch.tensor([0], dtype=torch.long)
    data_list.append(Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y))

for i in range(len(subject_1_plv['R'])):
    x = torch.tensor(subject_1_features['R'][i], dtype=torch.float)
    adj = torch.tensor(subject_1_plv['R'][i], dtype=torch.float)
    edge_index, edge_attr = dense_to_sparse(adj)
    y = torch.tensor([1], dtype=torch.long)
    data_list.append(Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y))


print(f"Total number of graphs (trials) created: {len(data_list)}")
print("\n--- Example Data Object ---")
print(data_list[0])
print("---------------------------")


# --- 2. Define the GAT Model Architecture ---

class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super(GAT, self).__init__()
        # We use GATv2Conv as it's a more powerful and stable version
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads, concat=True, edge_dim=1)
        self.gn1 = GraphNorm(hidden_channels * heads)
        
        self.conv2 = GATv2Conv(hidden_channels * heads, hidden_channels, heads=heads, concat=True, edge_dim=1)
        self.gn2 = GraphNorm(hidden_channels * heads)

        self.conv3 = GATv2Conv(hidden_channels * heads, hidden_channels, heads=heads, concat=True, edge_dim=1)
        self.gn3 = GraphNorm(hidden_channels * heads)
        
        # Final classifier layer
        self.lin = nn.Linear(hidden_channels * heads, out_channels)

    def forward(self, x, edge_index, edge_attr, batch, return_attention=False):
        # Layer 1
        x, att1 = self.conv1(x, edge_index, edge_attr, return_attention_weights=True)
        x = F.leaky_relu(x)
        x = self.gn1(x, batch)
        
        # Layer 2
        x, att2 = self.conv2(x, edge_index, edge_attr, return_attention_weights=True)
        x = F.leaky_relu(x)
        x = self.gn2(x, batch)
        
        # Layer 3
        x, att3 = self.conv3(x, edge_index, edge_attr, return_attention_weights=True)
        x = self.gn3(x, batch)
        
        # Readout layer for graph-level classification
        x = global_mean_pool(x, batch)
        
        # Apply dropout for regularization
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Final classification
        x = self.lin(x)
        
        if return_attention:
            # Return final logits and the attention weights from all layers
            return F.log_softmax(x, dim=1), (att1, att2, att3)
        else:
            return F.log_softmax(x, dim=1)

# --- Instantiate the model and print its summary ---
# in_channels = number of node features (8 frequency bands)
# hidden_channels = an intermediate dimension
# out_channels = number of classes (2: Left/Right)
# heads = number of parallel attention mechanisms
model = GAT(
    in_channels=data_list[0].num_node_features,  # now = 1 for MFF
    hidden_channels=16,
    out_channels=2,
    heads=4
)

print("\n--- GAT Model Architecture ---")
print(model)
print("----------------------------")